# 01 Training (protocol Sections 3-6)
Run the cells in order. Several Colab sessions can run this notebook at the same time (set a different `SESSION`); lock files prevent double runs and a restarted session continues where it stopped.
1. self-tests (must all print OK);  2. selection runs (45, at most 80 epochs);  3. factor selection and compute decision;  4. main runs A, B, C.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, zipfile, shutil, time
ROOT = '/content/drive/MyDrive/BUSI_QB'          # project folder on Drive
CODE = os.path.join(ROOT, 'code')                 # contains src/ (qsim.py, qbus.py, runner.py, prep.py) and protocols/
PREP = os.path.join(ROOT, 'prepared')
sys.path.insert(0, os.path.join(CODE, 'src'))
os.makedirs(PREP, exist_ok=True)
print(sorted(os.listdir(ROOT)))
SESSION = 'A'                                     # change to 'B', 'C' ... in a second session
RUNS = os.path.join(ROOT, 'runs')
import numpy as np, pandas as pd, tensorflow as tf
import qsim, qbus, runner
print(qbus.env_info())
assert tf.config.list_physical_devices('GPU'), 'select a GPU runtime'
try:
    tf.config.experimental.enable_op_determinism()
except Exception as e:
    print('determinism not available:', e)

In [ ]:
# 1. SELF-TESTS ------------------------------------------------------------------------------------------
rng = np.random.default_rng(0); ok = True
for n in (2, 4, 8):
    x = rng.uniform(-1, 1, (16, n)).astype(np.float32); w = rng.uniform(0, 2*np.pi, (2, n, 3)).astype(np.float32)
    for cnot in (True, False):
        a = qsim.circuit_expvals(x, w, qsim.NumpyOps, n, cnot)
        b = qsim.circuit_expvals(tf.constant(x), tf.constant(w), qsim.tf_ops(), n, cnot).numpy()
        ok &= np.abs(a - b).max() < 1e-5
        print(f'n={n} cnot={cnot} TF vs NumPy max diff {np.abs(a-b).max():.2e}')
    F = qsim.fourier_features(rng.uniform(-1, 1, (3 * 3**n + 50, n)), qsim.NumpyOps, n)
    print(f'n={n} Fourier features', F.shape[1])
# models, parameters, matched initialization, gradients
tab = []
for n in (2, 4, 8):
    ref = qbus.reference_init(n, 0)
    for b in ('q', 'qp', 's', 'tr', 'cl', 'id'):
        qbus.set_seeds(0); m = qbus.build_model(f'{b}{n}', k=10.0)
        d = qbus.apply_init(m, ref); ok &= d == 0.0
        g = qbus.gradient_share(m, (rng.random((64, 256, 256)) * 255).astype(np.uint8), (rng.random((64, 256, 256)) > 0.9).astype(np.uint8), 1)
        tab.append(dict(model=f'{b}{n}', **qbus.count_params(m), init_diff=d, grad_share=round(g, 6) if g == g else None))
for nm in ('full', 'unet'):
    m = qbus.build_model(nm); tab.append(dict(model=nm, **qbus.count_params(m)))
print(pd.DataFrame(tab).to_string(index=False))
# scaled weights: effective values equal k * stored values and the circuit init is uniform on [0, 2 pi)
m = qbus.build_model('q4', k=100.0); e = m.get_layer('block').effective()['angles']
ok &= (e.min() >= 0) and (e.max() < 2*np.pi + 1e-4); print('scaled circuit init range', e.min().round(3), e.max().round(3))
print('ALL SELF-TESTS OK' if ok else 'SELF-TEST FAILED: stop and report')

In [ ]:
# smoke test: one short run on 40 training images (writes to runs_smoke/, not analysed)
prep_ok = os.path.exists(os.path.join(PREP, 'busi_lesion.npz')); print('prepared data found:', prep_ok)
qbus.CFG_SAVE = dict(qbus.CFG); qbus.CFG.update(max_epochs=2, min_epochs=1)
d = np.load(os.path.join(PREP, 'busi_lesion.npz')); X, Y = d['x'][:60], d['y'][:60]
qbus.set_seeds(0); m = qbus.build_model('q4', k=10.0)
print(qbus.fit(m, X[:40], Y[:40], X[40:50], Y[40:50], seed=0, max_epochs=2))
print('test Dice', np.mean([r['dice'] for r in qbus.per_image(Y[50:], qbus.predict(m, X[50:]))]).round(3))
cr = qbus.code_replacement(m, X[:40], X[50:], Y[50:], n_draws=2); print({k: v.mean().round(3) for k, v in cr.items()})
print('MW', qbus.entanglement(m, X[50:]), 'surrogate R2', qbus.surrogate_fit(m, X[:40], X[50:]))
print('finite shots', qbus.finite_shot_dice(m, X[50:], Y[50:], repeats=1).mean().round(3))
qbus.CFG.update(qbus.CFG_SAVE); print('smoke test OK')

In [ ]:
# 2. SELECTION RUNS (fold 0, seed 100, k in {1, 10, 100}, at most 80 epochs) --------------------------------
SEL_DIR = os.path.join(RUNS, 'SEL')
for r in runner.run_list('SEL'):
    rid = runner.run_id(r)
    if runner.claim(SEL_DIR, rid, SESSION):
        print(time.strftime('%H:%M'), 'start', rid)
        s = runner.one_run(r, PREP, SEL_DIR, session=SESSION)
        print('   epochs', s['epochs'], 'best val loss', round(s['best_val_loss'], 4), 'min', round(s['minutes'], 1))

In [ ]:
# 3. FACTOR SELECTION AND COMPUTE DECISION (run once, after all 45 selection runs exist) ------------------
t, sel = runner.select_factors(SEL_DIR)
assert len(t) == 45, f'{len(t)} of 45 selection runs found'
print(t.pivot(index='model', columns='k', values='best_val_loss').round(4).to_string()); print('selected:', sel)
summ = [json.load(open(p)) for p in glob.glob(os.path.join(SEL_DIR, '*', 'summary.json'))]
per_epoch = pd.DataFrame([dict(model=s['model'], mpe=s['minutes'] / s['epochs'], ep=s['epochs']) for s in summ]).groupby('model').mean()
mean_ep = per_epoch.ep.mean()
def proj(plan, seeds_override=None):
    tot = 0.0
    for r in runner.run_list(plan):
        if seeds_override and r['seed'] not in seeds_override: continue
        mpe = per_epoch.mpe.get(r['model'], per_epoch.mpe.max())
        tot += mpe * 1.5 * mean_ep * (r['frac'] if r['frac'] < 1 else 1) ** 0.5
    return tot / 60
hA, hB, hC = proj('A'), proj('B'), proj('C')
decision = dict(selected=sel, hours=dict(A=hA, B=hB, C=hC), C_seeds=[0, 1], B_seeds=[0, 1, 2])
if hA + hB + hC > 80:
    decision['C_seeds'] = [0]; hC = proj('C', [0])
    if hA + hB + hC > 80:
        decision['B_seeds'] = [0, 1]
decision['time'] = time.strftime('%Y-%m-%d %H:%M:%S %Z')
p = os.path.join(RUNS, 'plan_decision.json')
if not os.path.exists(p):
    json.dump(decision, open(p, 'w'), indent=1)
print(json.load(open(p)))

In [ ]:
# 4. MAIN RUNS (A, then B, then C) ---------------------------------------------------------------------------
dec = json.load(open(os.path.join(RUNS, 'plan_decision.json'))); SELECTION = dec['selected']
for plan in ('A', 'B', 'C'):
    keep = dec.get(f'{plan}_seeds')
    for r in runner.run_list(plan):
        if keep and r['seed'] not in keep: continue
        rid = runner.run_id(r); out = os.path.join(RUNS, plan)
        if runner.claim(out, rid, SESSION):
            print(time.strftime('%m-%d %H:%M'), 'start', rid)
            s = runner.one_run(r, PREP, out, selection=SELECTION, session=SESSION)
            print('   epochs', s['epochs'], 'Dice', round(s.get('test_dice', float('nan')), 4), 'min', round(s['minutes'], 1))

In [ ]:
# progress
for plan in ('SEL', 'A', 'B', 'C'):
    done = len(glob.glob(os.path.join(RUNS, plan, '*', 'summary.json')))
    locks = len(glob.glob(os.path.join(RUNS, plan, '*.lock')))
    print(plan, 'done', done, 'running', locks)